# B07：小型 MLP 与训练诊断（辅修学生练习）

2 实验学时。完整前向与训练器已提供；补关键梯度及更新，固定 200 步，比较学习率 0 的故障与 0.1 修复。只改变学习率，不搜索轮数/容量。

## 1. 先写诊断预期

学习率 0 且梯度非零时，参数、损失会怎样？改为 0.1 是否保证训练/验证都下降？

**我的预期及理由：**（运行前填写）

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold

candidates = []
for parent in (Path.cwd(), *Path.cwd().parents):
    candidates.extend(
        [
            parent / "data/training-classification.csv",
            parent / "units/mlp-training/data/training-classification.csv",
        ]
    )
data_path = next((p for p in candidates if p.is_file()), None)
if data_path is None:
    raise FileNotFoundError("请保留目录结构和labs/mlp_core.py")
sys.path.insert(0, str(data_path.parent.parent / "labs"))
from mlp_core import initialize, forward, loss, gradients, train

raw = pd.read_csv(data_path)
dev = raw[raw.split == "train"].reset_index(drop=True)
features = ["planned_load_ratio", "ambient_index"]
ti, vi = list(
    StratifiedKFold(n_splits=3, shuffle=True, random_state=42).split(
        dev[features], dev.over_limit
    )
)[0]
preprocessor = Pipeline(
    [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
)
preprocessor.fit(dev.iloc[ti][features])
X = preprocessor.transform(dev.iloc[ti][features])
y = dev.iloc[ti].over_limit.to_numpy().reshape(-1, 1)
V = preprocessor.transform(dev.iloc[vi][features])
target_valid = dev.iloc[vi].over_limit.to_numpy().reshape(-1, 1)
initial = initialize()
assert X.shape == (80, 2) and y.shape == (80, 1) and V.shape == (40, 2)
print(
    "训练/验证形状：",
    X.shape,
    V.shape,
    "输出：",
    forward(X, initial)["probability"].shape,
)

## 2. 两处关键代码

平均二分类交叉熵对输出分数的梯度为(P−y)/n。后续矩阵积和求和已由 gradients 提供，不再重复除 n。输出与标签均(n,1)。

In [ ]:
def output_delta(probability, y):
    if probability.shape != y.shape or y.ndim != 2 or y.shape[1] != 1 or not len(y):
        raise ValueError("probability and y must share nonempty (n,1) shape")
    # TODO：返回平均损失对分数的梯度，删除停止提示。
    raise NotImplementedError("请完成输出梯度")
    return (... - ...) / len(y)

update(params,grad,learning_rate)返回新字典，各参数减去学习率乘同名梯度。所有梯度已按旧参数算完，不原地修改 params。

In [ ]:
def update(params, grad, learning_rate):
    # TODO：返回更新后的新字典，删除停止提示。
    raise NotImplementedError("请完成梯度下降更新")
    return {key: params[key] - ... * grad[key] for key in params}

In [ ]:
# 两条样本的已知结果，用于核对方向、批量平均和输出形状。
check_probability = np.array([[0.75], [0.25]])
check_y = np.array([[0.0], [1.0]])
check_delta = output_delta(check_probability, check_y)
assert check_delta.shape == (2, 1), "输出梯度应为 (2, 1)"
np.testing.assert_allclose(check_delta, [[0.375], [-0.375]], rtol=0, atol=1e-12)

saved = {key: value.copy() for key, value in initial.items()}
grad = gradients(X, y, initial, output_delta)
gradient_norm = np.sqrt(sum(np.sum(value**2) for value in grad.values()))
one_step = update(initial, grad, 0.1)
for key in initial:
    assert grad[key].shape == initial[key].shape
    np.testing.assert_array_equal(initial[key], saved[key])
    np.testing.assert_allclose(one_step[key] - initial[key], -0.1 * grad[key])
print("初始梯度范数：", gradient_norm)
print(
    "单步训练损失：",
    loss(forward(X, initial)["score"], y),
    "→",
    loss(forward(X, one_step)["score"], y),
)
assert gradient_norm > 0

## 3. 运行冻结与修复（已提供）

每次从同一组初始参数开始。第 0 步是更新前，第 200 步是预定终点。每一步的验证损失只用于记录。运行结束后，从修复组的记录中找出验证损失最低的一步，与第 200 步对照。

In [ ]:
curve_frames = []
summary_rows = []
models = {}
for name, rate in [("frozen", 0.0), ("repaired", 0.1)]:
    params, history = train(
        X, y, V, target_valid, initial, rate, output_delta, update, steps=200
    )
    models[name] = params
    frame = pd.DataFrame(history)
    frame.insert(0, "run", name)
    curve_frames.append(frame)
    difference = np.sqrt(sum(np.sum((params[k] - initial[k]) ** 2) for k in initial))
    summary_rows.append(
        {
            "run": name,
            "learning_rate": rate,
            "initial_train_loss": history[0]["train_loss"],
            "final_train_loss": history[-1]["train_loss"],
            "final_validation_loss": history[-1]["validation_loss"],
            "parameter_change_norm": difference,
        }
    )
curves = pd.concat(curve_frames, ignore_index=True)
summary = pd.DataFrame(summary_rows)
display(summary)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
for ax, (name, frame) in zip(axes, curves.groupby("run", sort=False)):
    ax.plot(frame.epoch, frame.train_loss, label="train")
    ax.plot(frame.epoch, frame.validation_loss, label="validation")
    ax.set(title=name, xlabel="Update count", ylabel="Mean binary cross-entropy")
    ax.legend()
fig.tight_layout()
plt.show()
for key in initial:
    np.testing.assert_array_equal(models["frozen"][key], initial[key])
    np.testing.assert_array_equal(initial[key], saved[key])
assert summary.set_index("run").loc["repaired", "parameter_change_norm"] > 0

repaired = curves.loc[curves["run"] == "repaired"].set_index("epoch")
best_step = repaired["validation_loss"].idxmin()
comparison = repaired.loc[[best_step, 200], ["train_loss", "validation_loss"]].copy()
comparison.insert(0, "观察点", ["验证损失最低", "预定终点"])
display(comparison.round(6))

## 4. 诊断说明

1. 引用配置、梯度范数、参数变化和曲线，说明本例为什么冻结。只有平曲线能唯一定位吗？
2. 修复组在哪一步记录到最低验证损失？比较该步和第 200 步的训练与验证损失，说明继续更新对两者的影响。若据此选定某一步作为最终模型，还需要什么评价数据？本实验能否证明 0.1 是最优学习率？
3. 梯度中的除以 n 来自哪里？为什么全部梯度应先于更新计算？
4. 验证集在哪一步使用，为什么没有参与反向传播？

**我的解释：**（填写）

重启顺序运行，保存本Notebook到非保护盘并备份。原B06项目不强制加网络；T15再组织评价与冻结。